In [0]:
CREATE OR REPLACE TABLE production.supply_analytics.dst_booking_dataset AS

-- ============================================
-- PARAMETERS - UPDATE THESE
-- ============================================
WITH params AS (
  SELECT 
    DATE '2025-01-01' AS yoy_pre_start_date,
    DATE '2025-01-15' AS yoy_pre_end_date,
    DATE '2025-01-16' AS yoy_event_week_start,
    DATE '2025-01-30' AS yoy_post_end_date,
    DATE '2026-01-01' AS event_pre_start_date,
    DATE '2026-01-15' AS event_pre_end_date,
    DATE '2026-01-16' AS event_week_start_str,
    DATE '2026-01-30' AS event_post_end_date,
    ARRAY('Germany') AS countries,
    ARRAY() AS tour_ids
),

-- ============================================
-- Base booking data with period windows
-- ============================================
booking_base AS (
  SELECT
    fb.booking_id,
    fb.tour_id,
    tour.user_id AS supplier_id,
    dl.country_name,
    fb.tickets,
    fb.nr,
    fb.gmv,
    fb.supplier_share,
    fb.gmv_supplier,
    CAST(fb.date_of_checkout AS date) AS dt_checkout,
    CASE
      WHEN CAST(fb.date_of_checkout AS date) BETWEEN params.yoy_pre_start_date AND params.yoy_pre_end_date THEN 'PRE_LY'
      WHEN CAST(fb.date_of_checkout AS date) BETWEEN params.yoy_event_week_start AND params.yoy_post_end_date THEN 'POST_LY'
      WHEN CAST(fb.date_of_checkout AS date) BETWEEN params.event_pre_start_date AND params.event_pre_end_date THEN 'PRE_CY'
      WHEN CAST(fb.date_of_checkout AS date) BETWEEN params.event_week_start_str AND params.event_post_end_date THEN 'POST_CY'
    END AS period_window
  FROM production.dwh.fact_booking_v2 AS fb
  CROSS JOIN params
  INNER JOIN production.dwh.dim_status AS s
    ON COALESCE(fb.status_id, -1) = s.status_id
  LEFT JOIN production.dwh.dim_tour AS tour
    ON fb.tour_id = tour.tour_id
  LEFT JOIN production.dwh.dim_location AS dl
    ON tour.location_id = dl.location_id
  WHERE
    s.status_display = 'Active'
    AND (SIZE(params.countries) = 0 OR ARRAY_CONTAINS(params.countries, dl.country_name))
    AND (
      CAST(fb.date_of_checkout AS date) BETWEEN params.yoy_pre_start_date AND params.yoy_post_end_date
      OR CAST(fb.date_of_checkout AS date) BETWEEN params.event_pre_start_date AND params.event_post_end_date
    )
    AND (
      SIZE(params.tour_ids) = 0
      OR fb.tour_id IN (SELECT tour_id FROM params LATERAL VIEW explode(params.tour_ids) e AS tour_id)
    )
)

-- ============================================
-- Aggregate bookings by country and period window
-- ============================================
SELECT
  country_name,
  period_window,
  COUNT(DISTINCT booking_id)                        AS bookings,
  COUNT(DISTINCT tour_id)                           AS total_tours,
  COUNT(DISTINCT supplier_id)                       AS total_suppliers,
  SUM(tickets)                                      AS tickets,
  SUM(nr)                                           AS nr,
  SUM(gmv)                                          AS gmv,
  SUM(supplier_share)                               AS supplier_share_revenue,
  SUM(gmv_supplier)                                 AS gmv_supplier,
  SUM(gmv) / NULLIF(SUM(tickets), 0)               AS avg_gmv_per_ticket,
  SUM(nr)  / NULLIF(SUM(tickets), 0)               AS avg_nr_per_ticket,
  SUM(gmv) / NULLIF(COUNT(DISTINCT booking_id), 0) AS avg_gmv_per_booking,
  SUM(nr)  / NULLIF(COUNT(DISTINCT booking_id), 0) AS avg_nr_per_booking
FROM booking_base
WHERE period_window IS NOT NULL
GROUP BY country_name, period_window
ORDER BY
  country_name,
  CASE period_window
    WHEN 'PRE_LY'  THEN 1
    WHEN 'POST_LY' THEN 2
    WHEN 'PRE_CY'  THEN 3
    WHEN 'POST_CY' THEN 4
  END;
